# EEG Preprocessing Pipeline Tutorial

## Part of the open_dvm Toolbox

This tutorial demonstrates how to preprocess raw EEG data using the open_dvm toolbox. We cover data loading, artifact correction, filtering, and ICA-based component removal to prepare data for subsequent ERP, TFR, BDM, and CTF analyses.

## Learning Objectives

After completing this tutorial, you will:

- **Understand the open_dvm preprocessing workflow** — Load raw EEG and behavioral data
- **Configure preprocessing parameters** — Set filtering, ICA, and artifact rejection settings
- **Handle subject-specific issues** — Define bad channels per subject
- **Integrate eye-tracking data** — Exclude saccades and fixation breaks
- **Process behavioral metadata** — Align behavioral events with EEG data
- **Verify preprocessed data** — Check data quality and trial counts
- **Save processed epochs** — Create publication-ready datasets for analysis

**Prerequisites:** Raw EEG data and behavioral logs from your experiment. This tutorial uses example data from a visual search task with 7 subjects.

## Overview: The Preprocessing Pipeline

### Data Flow

Raw EEG → Load & Montage → Filter → ICA → Autoreject → Epoch → Eye-Tracking QC → Save

### Key Steps

1. **Load raw EEG** — Import continuous EEG data from BioSemi or other formats
2. **Apply montage** — Define electrode positions (e.g., BioSemi-32)
3. **Filter data** — High-pass (remove drift) and optch (remove line noise)
4. **Run ICA** — Independent Component Analysis to identify and remove artifacts
5. **Apply autoreject** — Automatic bad segment detection and interpolation
6. **Create epochs** — Segment continuous data around trigger events
7. **Eye-tracking QC** — Exclude trials with saccades or fixation breaks
8. **Update metadata** — Add behavioral variables to each epoch
9. **Save epochs** — Store preprocessed data for analysis

## Section 1: Setup and Configuration

### 1.1 Import Required Libraries

In [3]:
# Add open_dvm package to path (temporary until we create and install the package)
import sys
import os
sys.path.insert(0, '/Users/dvm/Documents/DvM')

# Import open_dvm modules
import warnings
warnings.filterwarnings('ignore')

from open_dvm.analysis.preprocessing_pipeline import eeg_preprocessing_pipeline
from open_dvm.support.FolderStructure import FolderStructure
from open_dvm.support.preprocessing_utils import format_subject_id

print("✓ open_dvm preprocessing module imported successfully!")

✓ open_dvm preprocessing module imported successfully!


### 1.2 Define Project Structure and Paths

Set the root project folder and initialize the FolderStructure tracker. This defines where raw data, preprocessed data, and behavioral files are stored.

In [4]:
# ============================================
# CHANGE THIS PATH TO YOUR DATA LOCATION
# ============================================
project_folder = '/Users/dvm/Library/CloudStorage/OneDrive-VrijeUniversiteitAmsterdam/projects/openDvM'
os.chdir(project_folder)

# Initialize FolderStructure to manage file organization
fs = FolderStructure()

print(f'✓ Project folder set to: {project_folder}')
print(f'✓ FolderStructure initialized')

✓ Project folder set to: /Users/dvm/Library/CloudStorage/OneDrive-VrijeUniversiteitAmsterdam/projects/openDvM
✓ FolderStructure initialized


### 1.3 Define Subject-Specific Information

Specify any known bad channels for each subject (e.g., from electrode contact issues). These channels will be interpolated during preprocessing.

In [5]:
# Subject-specific information (bad channels identified during recording)
sj_info = {
    '1': {'bad_chs': []},              # Subject 1: no bad channels
    '2': {'bad_chs': ['C4', 'CP2']},   # Subject 2: C4 and CP2 had contact issues
    '3': {'bad_chs': ['C4', 'CP2']},
    '4': {'bad_chs': ['P3', 'P7']},
    '5': {'bad_chs': []},
    '6': {'bad_chs': []},
    '7': {'bad_chs': []}
}

print('Subject-specific bad channels:')
for sj, info in sj_info.items():
    if info['bad_chs']:
        print(f'  Subject {sj}: {info["bad_chs"]}')
    else:
        print(f'  Subject {sj}: none')

Subject-specific bad channels:
  Subject 1: none
  Subject 2: ['C4', 'CP2']
  Subject 3: ['C4', 'CP2']
  Subject 4: ['P3', 'P7']
  Subject 5: none
  Subject 6: none
  Subject 7: none


## Section 2: Preprocessing Parameters

### 2.1 EEG Recording Parameters

Define the characteristics of the EEG recording (sampling rate, number of sessions, electrode montage, etc.).

In [ ]:
# EEG recording setup
montage = 'biosemi32'              # Electrode montage
nr_sessions = 1                    # Number of independent EEG sessions
nr_sjs = 7                         # Number of subjects
eeg_runs = [1]                     # Recording runs within session (list for multiple runs)

# Channel information
eog = ['V_up', 'V_do', 'H_r', 'H_l']  # Eye-tracking channels (vertical up/down, horizontal right/left)
ref = ['Ref_r', 'Ref_l']                # Reference channels (BioSemi separate references)

print(f'Recording configuration:')
print(f'  Montage: {montage}')
print(f'  Sessions: {nr_sessions}')
print(f'  Subjects: {nr_sjs}')
print(f'  Runs per session: {eeg_runs}')
print(f'  EOG channels: {eog}')
print(f'  Reference channels: {ref}')

### 2.2 Epoching Parameters

Define how to extract epochs from continuous EEG data around trigger events.

In [ ]:
# Epoching setup
trigger_header = 'display_trigger'  # Behavioral column containing trigger codes
event_id = list(range(1, 250))      # Trigger codes to epoch (1-249)
t_min = -0.2                        # Time before trigger (seconds)
t_max = 0.5                         # Time after trigger (seconds)
flt_pad = 0.5                       # Epoch extension to control for filter artifacts

print(f'Epoching configuration:')
print(f'  Trigger header: {trigger_header}')
print(f'  Event codes: {len(event_id)} triggers ({min(event_id)}-{max(event_id)})')
print(f'  Epoch window: [{t_min:.1f}, {t_max:.1f}] seconds')
print(f'  Filter padding: {flt_pad} seconds')

### 2.3 Filtering and Artifact Correction Parameters

Specify high-pass filtering, notch filtering, ICA, and autoreject settings.

In [ ]:
# Preprocessing parameter dictionary
preproc_param = {
    'high_pass': 0.01,      # High-pass filter cutoff (Hz) - removes DC drift
    'run_ica': True,        # Run Independent Component Analysis
    'run_autoreject': True, # Run automatic artifact rejection
    'notch': True           # Apply notch filter at 50 Hz (line noise)
}

print('Preprocessing parameters:')
for key, value in preproc_param.items():
    print(f'  {key}: {value}')

### 2.4 Behavioral Data Selection

Define which behavioral columns to include in the epoch metadata.

In [ ]:
# Behavioral columns to include in epoch metadata
beh_oi = [
    'nr_trials', 'display_trigger', 'RT', 'subject_nr', 'block_cnt',
    'practice', 'block_type', 'correct', 'dist_loc', 'target_loc',
    'dist_cnd', 'dist_img', 'neut_shape', 'target_cnd', 'high_prob_dist'
]

print(f'Behavioral variables to include ({len(beh_oi)}):') 
for var in beh_oi:
    print(f'  - {var}')

### 2.5 Eye-Tracking Quality Control

Define parameters for excluding trials with eye movement artifacts (saccades, fixation breaks).

In [ ]:
# Eye-tracking information and quality control
eye_info = {
    'tracker_ext': 'asc',           # File extension for eye tracker data
    'sfreq': 1000,                  # Eye tracker sampling rate (Hz)
    'trigger_msg': 'Onset search',  # Message marking trial start in eye tracking data
    'window_oi': (-700, 1000),      # Time window around trigger (ms)
    'start': 'start trial',         # Event marking trial onset
    'drift_correct': None,          # Drift correction method (None = none)
    'viewing_dist': 70,             # Viewing distance (cm)
    'screen_res': (1920, 1080),     # Screen resolution (pixels)
    'screen_h': 29                  # Screen height (cm)
}

print('Eye-tracking information:')
for key, value in eye_info.items():
    print(f'  {key}: {value}')

## Section 3: Run the Preprocessing Pipeline

### 3.1 Preprocess a Single Subject

Run the complete preprocessing pipeline for one subject as an example. This step takes ~5-10 minutes per subject depending on data length and computational resources.

In [ ]:
# Preprocess subject 1 (example)
sj = 1

print(f'Starting preprocessing for subject {sj}...')
print('This may take several minutes...\n')

# Call the preprocessing pipeline
eeg_preprocessing_pipeline(
    sj=sj,
    session=1,
    eog=eog,
    ref=ref,
    eeg_runs=eeg_runs,
    nr_sessions=nr_sessions,
    t_min=t_min,
    t_max=t_max,
    flt_pad=flt_pad,
    sj_info=sj_info,
    eye_info=eye_info,
    event_id=event_id,
    montage=montage,
    preproc_param=preproc_param,
    trigger_header=trigger_header,
    beh_oi=beh_oi,
    binary=0,
    preproc_name='main',
    nr_sjs=nr_sjs,
    excl_factor=None
)

print(f'✓ Subject {sj} preprocessing complete!')

### 3.2 Batch Process All Subjects

Run preprocessing for all subjects in the dataset. Uncomment and modify below to process multiple subjects.

In [ ]:
# Uncomment below to process all subjects (optional)
# for sj in range(1, 8):  # Subjects 1-7
#     print(f'\n{'='*60}')
#     print(f'Processing subject {sj}...')
#     print('='*60)
#     
#     eeg_preprocessing_pipeline(
#         sj=sj,
#         session=1,
#         eog=eog,
#         ref=ref,
#         eeg_runs=eeg_runs,
#         nr_sessions=nr_sessions,
#         t_min=t_min,
#         t_max=t_max,
#         flt_pad=flt_pad,
#         sj_info=sj_info,
#         eye_info=eye_info,
#         event_id=event_id,
#         montage=montage,
#         preproc_param=preproc_param,
#         trigger_header=trigger_header,
#         beh_oi=beh_oi,
#         binary=0,
#         preproc_name='main',
#         nr_sjs=nr_sjs,
#         excl_factor=None
#     )
#     
#     print(f'✓ Subject {sj} complete')
# 
# print(f'\n{'='*60}')
# print('✓ All subjects preprocessing complete!')
# print('='*60)

## Section 4: Verify Preprocessed Data

### 4.1 Load and Inspect Preprocessed Epochs

In [ ]:
# Load the preprocessed data for inspection
sj = 1

eye_dict = {
    'use_tracker': True,
    'window_oi': (0, 0.3),
    'angle_thresh': 1,
    'viewing_dist': 70,
    'screen_res': (1920, 1080),
    'screen_h': 29,
    'drift_correct': (-0.2, 0)
}

df, epochs = FolderStructure().load_processed_epochs(
    sj, 'ses_01_main', 'main', eye_dict
)

print(f'✓ Data loaded for subject {sj}')
print(f'\nEpochs: {len(epochs)} trials')
print(f'Channels: {len(epochs.ch_names)}')
print(f'Sampling rate: {epochs.info["sfreq"]} Hz')
print(f'Time range: {epochs.tmin:.3f} to {epochs.tmax:.3f} s')
print(f'\nBehavioral data shape: {df.shape[0]} trials × {df.shape[1]} variables')
print(f'\nBehavioral variables:')
print(df.columns.tolist())

### 4.2 Check Trial Counts by Condition

In [ ]:
# Check trial counts across conditions
print('Trial counts by experimental conditions:\n')

print('By block type:')
print(df['block_type'].value_counts())

print('\nBy target condition:')
print(df['target_cnd'].value_counts())

print('\nBy distractor condition:')
print(df['dist_cnd'].value_counts())

print('\nCross-condition:')
print(df[['target_cnd', 'dist_cnd']].value_counts().head(10))

## Section 5: Summary and Next Steps

### What You've Accomplished

1. ✓ Set up the project folder and configuration
2. ✓ Defined subject-specific preprocessing parameters
3. ✓ Configured filtering, ICA, and artifact rejection
4. ✓ Integrated eye-tracking quality control
5. ✓ Ran the complete preprocessing pipeline
6. ✓ Verified the preprocessed epochs

### Output Files

The preprocessing pipeline saves:
- **Preprocessed epochs**: `eeg/processed/sub_*_ses_1_main-epo.fif`
- **Behavioral metadata**: `behavioral/processed/sub_*_ses_1.csv`
- **ICA components**: `eeg/processed/sub_*_ses_1_main-ica.fif`
- **Quality reports**: `eeg/reports/sub_*_ses_1_main.html`

### Next Steps

Now that you have preprocessed EEG data, you can proceed to:

1. **ERP Analysis** → See `02_erp_analysis.ipynb` for computing event-related potentials
2. **TFR Analysis** → Time-frequency analysis of oscillatory activity
3. **BDM Analysis** → Multivariate classification/decoding analysis
4. **CTF Analysis** → Cross-temporal decoding (generalization across time)

All subsequent analyses use the preprocessed epochs loaded via `FolderStructure().load_processed_epochs()`.